In [1]:
# =======================
# MODELBUILDER: ROBUST LOADER + NORMALIZATION
# =======================
import pandas as pd, numpy as np
from pathlib import Path

PROC = Path("./processed")
candidates = [
    PROC/"quarterly_features.csv",          # canonical
    PROC/"quarterly_features_clean.csv",    # legacy
    PROC/"quarterly_features.parquet"       # parquet
]

def load_quarterly_features():
    for p in candidates:
        if p.suffix == ".csv" and p.exists():
            try:
                # First try: CSV has an explicit date_q column (our new standard)
                df = pd.read_csv(p, parse_dates=["date_q"])
                df = df.set_index("date_q").sort_index()
                print(f"[LOAD] {p.name} (CSV with date_q) → {df.shape}")
                return df
            except Exception:
                # Fallback: CSV with the date in the first unnamed column (older runs)
                df = pd.read_csv(p)
                # Try to parse the first column if it looks like a date column
                first_col = df.columns[0]
                try:
                    df[first_col] = pd.to_datetime(df[first_col], errors="coerce")
                    if df[first_col].notna().sum() > 0:
                        df = df.set_index(first_col).sort_index()
                        df.index.name = "date_q"
                        print(f"[LOAD] {p.name} (CSV legacy index) → {df.shape}")
                        return df
                except Exception:
                    pass
        elif p.suffix == ".parquet" and p.exists():
            df = pd.read_parquet(p)
            # Parquet saved with index in FeatureBuilder cell above
            if "date_q" in df.index.names or isinstance(df.index, pd.DatetimeIndex):
                df = df.sort_index()
                if df.index.name is None:
                    df.index.name = "date_q"
                print(f"[LOAD] {p.name} (Parquet) → {df.shape}")
                return df
            # If parquet had date_q as a column:
            if "date_q" in df.columns:
                df["date_q"] = pd.to_datetime(df["date_q"], errors="coerce")
                df = df.set_index("date_q").sort_index()
                print(f"[LOAD] {p.name} (Parquet with date_q col) → {df.shape}")
                return df
    raise FileNotFoundError("No quarterly features file found. Expected one of: " + ", ".join([str(x) for x in candidates]))

qdf = load_quarterly_features()

# Ensure numeric for downstream
for c in qdf.columns:
    qdf[c] = pd.to_numeric(qdf[c], errors="coerce")

# Minimal required columns — create if missing (safe guards)
if "excess_ret" not in qdf and {"midcap_qret","nifty_qret"}.issubset(qdf.columns):
    qdf["excess_ret"] = qdf["midcap_qret"] - qdf["nifty_qret"]

if "ret_prev_q" not in qdf and "excess_ret" in qdf:
    qdf["ret_prev_q"] = qdf["excess_ret"].shift(1)

for base, lag in [("rain_anom","rain_anom_lag"),
                  ("cpi_yoy","cpi_yoy_lag"),
                  ("gdp_yoy","gdp_yoy_lag"),
                  ("repo_chg_bps","repo_chg_lag")]:
    if (lag not in qdf) and (base in qdf):
        qdf[lag] = qdf[base].shift(1)

if ("excess_next_q" not in qdf) and ("excess_ret" in qdf):
    qdf["excess_next_q"] = qdf["excess_ret"].shift(-1)

print("ModelBuilder ready — rows:", len(qdf))

[LOAD] quarterly_features.csv (CSV with date_q) → (41, 6)
ModelBuilder ready — rows: 41


RQ1 — Baseline vs Enriched

In [2]:
# Expect: results_rq1 (DataFrame of metrics), preds_rq1 (OOF predictions DataFrame with columns: y_true, y_pred_baseline, y_pred_enriched)
RESULTS_DIR = Path("./processed"); RESULTS_DIR.mkdir(exist_ok=True)

# 1) Metrics
rq1_metrics = results_rq1.copy()
rq1_metrics.to_csv(RESULTS_DIR/"rq1_metrics.csv", index=True)
print("Saved:", (RESULTS_DIR/"rq1_metrics.csv").resolve())

# 2) OOF predictions for the scatter/actual-vs-predicted charts
# Ensure a clean, minimal schema:
#   date_q (index) | y_true | y_pred_baseline | y_pred_enriched
if "date_q" not in preds_rq1.columns and preds_rq1.index.name == "date_q":
    preds_rq1 = preds_rq1.reset_index()
preds_rq1.to_csv(RESULTS_DIR/"rq1_oof_predictions.csv", index=False, float_format="%.6f")
print("Saved:", (RESULTS_DIR/"rq1_oof_predictions.csv").resolve())


NameError: name 'results_rq1' is not defined

RQ2 — Good vs Poor Monsoon

In [ ]:
# Expect: groups summary + stats objects already computed
# Build two small outputs:
#   (a) monsoon_groups, per-quarter classification
#   (b) test summary: means, deltas, t/KS stats

monsoon_groups = monsoon_groups[['rain_group','excess_next_q']].copy()  # ensure tidy
monsoon_groups.to_csv(RESULTS_DIR/"rq2_monsoon_groups.csv", index=True)
print("Saved:", (RESULTS_DIR/"rq2_monsoon_groups.csv").resolve())

rq2_summary = pd.DataFrame({
    "N_good":[N_good], "N_poor":[N_poor],
    "mean_good":[mean_good], "mean_poor":[mean_poor],
    "delta_mean":[mean_good-mean_poor],
    "t_stat":[t_stat], "t_pvalue":[t_pvalue],
    "ks_D":[ks_D], "ks_pvalue":[ks_pvalue]
})
rq2_summary.to_csv(RESULTS_DIR/"rq2_good_vs_poor_rain.csv", index=False, float_format="%.6f")
print("Saved:", (RESULTS_DIR/"rq2_good_vs_poor_rain.csv").resolve())

RQ3 — Rain → GDP(t+1) & forecast uplift

In [ ]:
# Expect: rq3_tbl (metrics with/without gdp_pred_from_rain)
rq3_tbl.to_csv(RESULTS_DIR/"rq3_enriched_with_gdp_pred.csv", index=True, float_format="%.6f")
print("Saved:", (RESULTS_DIR/"rq3_enriched_with_gdp_pred.csv").resolve())

# (Optional) If you created the OLS summary as variables:
# e.g., beta_hat, beta_p, ci_lo, ci_hi — capture them as a one-row table:
try:
    rq3_ols = pd.DataFrame([{
        "beta_rain": beta_hat, "p_value": beta_p,
        "ci_low": ci_lo, "ci_high": ci_hi
    }])
    rq3_ols.to_csv(RESULTS_DIR/"rq3_ols_rain_to_gdp.csv", index=False, float_format="%.6f")
    print("Saved:", (RESULTS_DIR/'rq3_ols_rain_to_gdp.csv').resolve())
except NameError:
    pass


RQ4 — Drivers & Interactions

In [ ]:
# Expect: perm_imp_df (columns: feature, importance), rq4_int_tbl (metrics with/without interaction)
perm_imp_df.to_csv(RESULTS_DIR/"rq4_permutation_importance.csv", index=False, float_format="%.6f")
print("Saved:", (RESULTS_DIR/"rq4_permutation_importance.csv").resolve())

rq4_int_tbl.to_csv(RESULTS_DIR/"rq4_interaction_uplift.csv", index=True, float_format="%.6f")
print("Saved:", (RESULTS_DIR/"rq4_interaction_uplift.csv").resolve())
